In [1]:
from argilla_client import client, rg, workspace


In [2]:
import json
from pathlib import Path

from pydantic import BaseModel


class User(BaseModel):
    username: str
    role: str
    first_name: str
    last_name: str
    password: str

    def to_argilla_user(self) -> rg.User:
        return rg.User(
            **self.model_dump(),
            client=client,
        )

notebook_path = Path(globals()["__vsc_ipynb_file__"]).resolve()
users_path = notebook_path.parent / "argilla_users.json"
users_list = [User.model_validate(item).to_argilla_user() for item in json.loads(users_path.read_text())]

for user in users_list:
    existing = client.users(user.username)
    if existing is not None and existing.username != client.me.username:
        existing.delete()
    user.create()
    user.add_to_workspace(workspace)